# 🏙️ The Chinese Decentralization Problem Set  
*(based on Baum-Snow et al., 2017)*

## Overview

This problem set explores how urban infrastructure—especially **radial roads and ring roads**—has shaped **urban population growth in China** between 1990 and 2010.  
Our data are derived from *Baum-Snow, Brandt, Henderson, Turner, and Zhang (2017), “Does Urban Rail Transit Investment Encourage Urban Growth? Evidence from China,”* NBER Working Paper 24596.

The dataset, `china_decent_regs.parquet`, contains variables at the **prefecture-city** level that measure changes in population, GDP, and infrastructure across time.  
We will replicate **Table 4** from the paper using Python, which reports **instrumental variables (IV) regressions** of population growth on road infrastructure.

---

## Learning Goals

By completing this exercise, you will:

1. Reinforce how to specify and interpret an **instrumental variables regression** in Python.  
2. Learn how to identify **endogenous** regressors and their corresponding **instruments**.  
3. Gain additional experience with **clustered standard errors** and **first-stage diagnostics** (e.g., weak instrument tests).  

---

## Core Research Question

> **Did the expansion of radial and ring roads cause faster population growth in Chinese cities?**

Formally, we estimate models of the form:

$$
\Delta \ln(\text{Population}_{i,1990\!-\!2010}) = 
\alpha + \beta \cdot \text{Roads}_{i,2010} + X_i'\gamma + \varepsilon_i,
$$

where:
- $i$ indexes prefecture-level cities,
- $\text{Roads}_{i,2010}$ measures the extent of roads by 2010,
- $X_i$ is a vector of baseline controls,
- $\varepsilon_i$ is an error term clustered by province.

Because road placement may be **endogenous** (e.g., more developed cities built more roads), we instrument for modern road measures using **historical road plans from 1962**:

$$
\text{Roads}_{i,2010} = \pi_0 + \pi_1 \cdot \text{Roads}_{i,1962} + X_i'\rho + u_i.
$$

The coefficient $\beta$ from the second stage represents the **causal effect** of road infrastructure on urban population growth.

---

## The Data

The dataset `china_decent_regs.parquet` (≈280 observations) is the product of extensive preparation combining:
- Historical maps of roads and railways (1924–2010),
- Census data (1982–2010),
- GDP and employment by sector,
- Geographic and institutional indicators.

---

### Relevant Variables

- `D_censuspop9010_cp` — Log change in core-prefecture (CP) population, 1990–2010  
- `D_censuspop9010_pref` — Log change in prefecture population, 1990–2010  
- `all_road_2010_rays` — Number of radial road connections by 2010  
- `road_1962_rays` — Number of planned radial road connections (1962)  
- `rail_2010_rays` — Number of radial railway connections by 2010  
- `rail_1962_rays` — Number of planned railway connections (1962)  
- `po_s_all_road_2010_ringxda` — Dummy = 1 if ring roads existed by 2010  
- `po_s_road_1962_ringxda` — Dummy = 1 if ring roads were planned in 1962  
- `lall_road_2010_km_pfcp` — Log kilometers of roads in the prefecture minus core area, 2010  
- `lroad_1962_km_pfcp` — Log kilometers of roads in the prefecture minus core area, 1962  
- `larea_cc90` — Log area of the central city (1990)  
- `larea_pf05` — Log area of the prefecture (2005)  
- `province_capitalPlus` — Dummy = 1 if provincial capital or provincial-level city  
- `lcensuspop1982_pref` — Log prefecture population, 1982  
- `frac_highedu1982_pref` — Fraction of adults with high school or higher education, 1982  
- `share_emp_man1982_pref` — Share of manufacturing employment, 1982  
- `ruralMigp00` — Share of rural migrants around 2000  
- `province05` — Province identifier (used for clustering)  
- `D0` — Sample inclusion dummy (1 = included in analysis)

---

**Next:** In the following code cells, you’ll load `china_decent_regs.parquet`, inspect key variables, and begin estimating the IV regressions.


# Question 1: Setup and Data Inspection
 
In this first part of the assignment, you will:
1. Import the necessary Python packages,
2. Define your base working directory,
3. Load the dataset `china_decent_regs.parquet`, and
4. Inspect the structure and summary statistics of the data.


In [1]:
# QUESTION 1A — Setup and Data Inspection

# load libraries os, pandas, numpy, statsmodels.api as sm, IV2SLS from linearmodels.iv
import os
from pathlib import Path
import pandas as pd
import numpy as np
import statsmodels.api as sm
try:
    from linearmodels.iv import IV2SLS
except ImportError:
    import pip
    print('installing \'linearmodels\'')
    pip.main(['install', 'linearmodels'])
    from linearmodels.iv import IV2SLS

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [9]:
# Define base directory (adjust this path as needed)
print(os.getcwd())  # Print current working directory to help set the correct path
# base_dir = Path("H:/Documents/GradSchool/UrbanEcon/Assignments")
base_dir = Path("C:/Users/nicho/Documents/World Building/MapCreator/file_transit")
# Load dataset
china_df = pd.read_parquet(base_dir / "china_decent_regs.parquet")
china_df.head()

c:\Users\nicho\Documents\World Building\MapCreator\file_transit


cc05  gdp_prefb1990  gdp_sector1_prefb1990  gdp_sector2_prefb1990  gdp_sector3_prefb1990  tot_pop_prefb1990  \
0  110000            NaN                    NaN                    NaN                    NaN                NaN   
1  120000            NaN                    NaN                    NaN                    NaN             870.46   
2  130100            NaN                    NaN                    NaN                    NaN                NaN   
3  130200            NaN                    NaN                    NaN                    NaN                NaN   
4  130300            NaN                    NaN                    NaN                    NaN                NaN   

   a101_prefb1990  a105_prefb1990  a106_prefb1990  a107_prefb1990  a108_prefb1990  a109_prefb1990  a110_prefb1990  \
0       3132264.0       1907873.0       6566875.0       3454936.0       3111939.0       1224391.0       4252539.0   
1       2551973.0       1481438.0       4814245.0       2464482.0       2349763.0       1061530.0       3914381.0   
2       2047062.0        425707.0       1591589.0        836107.0        755482.0       1617786.0       6431320.0   
3       1794478.0        378244.0       1331577.0        694348.0        637229.0       1364961.0       5062550.0   
4        689433.0        138160.0        481908.0        253272.0        228636.0        551243.0       1985196.0   

   a111_prefb1990  a112_prefb1990  a113_prefb1990  a114_prefb1990  a115_prefb1990  a116_prefb1990  a118_prefb1990  \
0       2138525.0       2114014.0             0.0             0.0             0.0             0.0       3377496.0   
1       1963265.0       1951116.0          9005.0         56801.0         42999.0         13802.0       2510218.0   
2       3243743.0       3187577.0          3569.0         32504.0         25747.0          6757.0        780475.0   
3       2573447.0       2489103.0         51273.0        196423.0        112083.0         84340.0        807477.0   
4       1016830.0        968366.0            30.0          2230.0          2194.0            36.0        261953.0   

   a119_prefb1990  a121_prefb1990  a122_prefb1990  a123_prefb1990  a124_prefb1990  a125_prefb1990  a126_prefb1990  \
0       2996021.0       2145037.0       2167951.0        132909.0         70928.0         61981.0        602782.0   
1       2297952.0       1936089.0       1990385.0         50783.0         24439.0         26344.0        331570.0   
2        611620.0       3305016.0       3317611.0         40691.0         20106.0         20585.0        531864.0   
3        619337.0       2564065.0       2582020.0         17651.0          8336.0          9315.0        643258.0   
4        215395.0       1006842.0        977680.0          7464.0          3501.0          3963.0        212135.0   

   a127_prefb1990  a128_prefb1990  a129_prefb1990  a130_prefb1990  a131_prefb1990  a132_prefb1990  a133_prefb1990  \
0       2047747.0       1045429.0       1002318.0        185247.0        605777.0        330123.0        275654.0   
1       1151750.0        590193.0        561557.0        139948.0        463151.0        249091.0        214060.0   
2       2108302.0       1075337.0       1032965.0         98717.0        401131.0        215435.0        185696.0   
3       2373706.0       1212179.0       1161527.0         93806.0        346338.0        185721.0        160617.0   
4        758195.0        389424.0        368771.0         34131.0        125646.0         66950.0         58696.0   

   a134_prefb1990  a135_prefb1990  a136_prefb1990  a137_prefb1990  a138_prefb1990  a139_prefb1990  a140_prefb1990  \
0        417535.0       1441970.0        715306.0        726664.0        595511.0        343312.0        252199.0   
1        191622.0        688599.0        341102.0        347497.0        363565.0        204849.0        158716.0   
2        433147.0       1707171.0        859902.0        847269.0        234613.0        142513.0         92100.0   
3        549452.0       2027

In [10]:

# Inspect structure
china_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Columns: 2888 entries, cc05 to D0
dtypes: float32(565), float64(1471), int16(20), int32(580), int8(250), object(2)
memory usage: 4.5+ MB


In [17]:

# Summary statistics for key variables
summary_vars = [
    "D_censuspop9010_cp", "all_road_2010_rays", "road_1962_rays",
    "larea_cc90", "larea_pf05", "province_capitalPlus",
    "lcensuspop1982_pref", "frac_highedu1982_pref",
    "share_emp_man1982_pref", "ruralMigp00"
]
summary_stats = china_df[summary_vars].describe()
print("--- Summary Statistics ---")
display(summary_stats)

--- Summary Statistics ---


,D_censuspop9010_cp,all_road_2010_rays,road_1962_rays,larea_cc90,larea_pf05,province_capitalPlus,lcensuspop1982_pref,frac_highedu1982_pref,share_emp_man1982_pref,ruralMigp00
count,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,286.000000,2.860000e+02
mean,0.396793,3.758741,1.989510,7.182302,9.348269,0.087413,14.791728,0.113018,0.113711,7.155524e+04
std,0.312836,1.968482,1.385348,0.963816,0.815393,0.282934,0.712620,0.045756,0.091725,1.400289e+05
min,-0.245528,0.000000,0.000000,4.631307,6.943104,0.000000,11.302204,0.024662,0.010871,0.000000e+00
25%,0.175775,2.000000,1.000000,6.471672,8.911930,0.000000,14.441394,0.082801,0.045622,1.062837e+04
50%,0.358078,4.000000,2.000000,7.303168,9.401667,0.000000,14.893425,0.109403,0.084208,3.115416e+04
75%,0.539127,5.000000,3.000000,7.878065,9.848189,0.000000,15.276958,0.133251,0.151041,7.542458e+04
max,1.749425,12.000000,6.000000,9.911949,12.027024,1.000000,17.110273,0.289761,0.512195,1.178439e+06


# Question 1B — Filter Relevant Variables

The raw dataset contains nearly 3,000 columns created for multiple tables in the original paper.  
For this problem set, we only need a small subset of variables used in Table 4.

In this step, you will:
1. Define the list of relevant variables (outcome, instruments, controls, and identifiers),  
2. Filter the DataFrame to keep only those variables, and  
3. Drop any observations missing the main outcome variable `D_censuspop9010_cp`.
4. Finally, keep ONLY the analysis sample where D0=1

After filtering, the dataset will be much smaller and easier to inspect.

In [16]:
# QUESTION 1B — Filter dataset to relevant variables

# Define the relevant variables for the Chinese Decentralization Problem Set (Table 4)
vars_needed = [
    # Outcome and instruments
    "D_censuspop9010_cp", "D_censuspop9010_pref",
    "all_road_2010_rays", "road_1962_rays",
    "rail_2010_rays", "rail_1962_rays",
    "po_s_all_road_2010_ringxda", "po_s_road_1962_ringxda",
    "lall_road_2010_km_pfcp", "lroad_1962_km_pfcp",
    
    # Controls
    "larea_cc90", "larea_pf05", "province_capitalPlus",
    "lcensuspop1982_pref", "frac_highedu1982_pref",
    "share_emp_man1982_pref", "ruralMigp00",
    
    # Clustering and sample
    "province05", "D0"
]

# Keep only those columns
working_china_df = china_df[vars_needed].copy()

# Drop rows missing the main outcome variable
working_china_df = working_china_df.dropna(subset=["D_censuspop9010_cp"])

# Keep analysis sample only
working_china_df = working_china_df[working_china_df["D0"] == 1]
print(f"{working_china_df.shape=}")
working_china_df.info() # Check the structure of the filtered dataset

working_china_df.shape=(257, 19)
<class 'pandas.core.frame.DataFrame'>
Index: 257 entries, 0 to 283
Data columns (total 19 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   D_censuspop9010_cp          257 non-null    float32
 1   D_censuspop9010_pref        257 non-null    float32
 2   all_road_2010_rays          257 non-null    int8   
 3   road_1962_rays              257 non-null    int8   
 4   rail_2010_rays              257 non-null    int8   
 5   rail_1962_rays              257 non-null    int8   
 6   po_s_all_road_2010_ringxda  257 non-null    float32
 7   po_s_road_1962_ringxda      257 non-null    float32
 8   lall_road_2010_km_pfcp      257 non-null    float32
 9   lroad_1962_km_pfcp          257 non-null    float32
 10  larea_cc90                  257 non-null    float32
 11  larea_pf05                  257 non-null    float32
 12  province_capitalPlus        257 non-null    float32
 13  lcensus

# Question 2 — OLS Baseline Regressions (Replicate Table 3)

In this section, you will estimate three OLS models that correspond to columns (1), (2), and (6) of Table 3 in Baum-Snow et al. (2017).

Each regression uses **Δln(CC Pop) 1990–2010** as the dependent variable:

1. **Column (1)** — Regress Δln(CC Pop) on **2010 radial highways** (no controls).  
2. **Column (2)** — Add **base controls** and **Δln(Pref Pop)** to account for prefecture-level population growth.  
3. **Column (6)** — Keep all controls and include the **2010 ring-road indicator** to test for additional effects of orbital infrastructure.

Use the already-filtered dataset (`df`, which includes only `D0 == 1`) and **cluster standard errors by province**.

After running the regressions, compare your coefficients with those in Table 3.  
- How does including Δln(Pref Pop) affect the coefficient on radial highways?  
- Does adding the ring-road indicator change the results substantially?


In [25]:
# QUESTION 2 — OLS Baseline Regressions (Replicate Table 3)

import statsmodels.formula.api as smf

# Use the filtered analysis sample created in Question 1B
df = working_china_df.copy()

cluster_var = "province05"
y_var = "D_censuspop9010_cp"

base_controls = [
    "larea_cc90",
    "larea_pf05",
    "province_capitalPlus",
    "lcensuspop1982_pref",
    "frac_highedu1982_pref",
    "share_emp_man1982_pref",
    "ruralMigp00",
]

def fit_ols_clustered(formula: str, data: pd.DataFrame, cluster_col: str):
    model = smf.ols(formula=formula, data=data, missing="drop")
    # Important: if rows are dropped due to missing values, clusters must match the used sample
    used_df = model.data.frame
    res = model.fit(cov_type="cluster", cov_kwds={"groups": used_df[cluster_col]})
    return res

# (1) Roads only
f1 = f"{y_var} ~ all_road_2010_rays"
ols_1 = fit_ols_clustered(f1, df, cluster_var)

# (2) Add base controls + Δln(Pref Pop)
f2 = f"{y_var} ~ all_road_2010_rays + D_censuspop9010_pref + " + " + ".join(base_controls)
ols_2 = fit_ols_clustered(f2, df, cluster_var)

# (6) Add ring-road indicator
f6 = f2 + " + po_s_all_road_2010_ringxda"
ols_6 = fit_ols_clustered(f6, df, cluster_var)

# Display compact coefficient tables
def compact_table(results_by_col: dict, variables: list[str]) -> pd.DataFrame:
    '''
       Construct a compact table of coefficients and standard errors for specified variables across multiple regression results. 
       results_by_col: dict mapping column name to regression result object
       variables: list of variable names to include in the table (in order)
    '''
    out = pd.DataFrame(index=variables)
    for col_name, res in results_by_col.items():
        coefs = res.params.reindex(variables)
        ses = res.bse.reindex(variables)
        out[col_name] = [
            f"{c:.4f} ({s:.4f})" if (pd.notna(c) and pd.notna(s)) else ""
            for c, s in zip(coefs, ses)
        ]
    return out

# Prepare results for display
results = {"(1)": ols_1, "(2)": ols_2, "(6)": ols_6}
vars_to_show = [
    "all_road_2010_rays",
    "D_censuspop9010_pref",
    "po_s_all_road_2010_ringxda",
    *base_controls,
    "Intercept",
]

# Statsmodels names the constant as 'Intercept' in the formula API
tbl = compact_table(results, vars_to_show)

# Add model diagnostics at bottom
diag = pd.DataFrame({
    k: {
        "N": int(v.nobs),
        "R2": float(v.rsquared),
    }
    for k, v in results.items()
}).T
display(tbl)
display(diag)

# Full regression output (optional)
print(ols_1.summary())
print(ols_2.summary())
print(ols_6.summary())

,(1),(2),(6)
all_road_2010_rays,0.0097 (0.0088),-0.0104 (0.0048),-0.0098 (0.0053)
D_censuspop9010_pref,,0.9974 (0.0615),0.9972 (0.0616)
po_s_all_road_2010_ringxda,,,0.0247 (0.0330)
larea_cc90,,-0.1155 (0.0186),-0.1108 (0.0208)
larea_pf05,,0.0537 (0.0206),0.0531 (0.0202)
province_capitalPlus,,0.0880 (0.0535),0.0858 (0.0543)
lcensuspop1982_pref,,0.0968 (0.0277),0.0946 (0.0276)
frac_highedu1982_pref,,-0.4570 (0.3563),-0.4672 (0.3586)
share_emp_man1982_pref,,-0.1676 (0.1963),-0.1712 (0.1972)
ruralMigp00,,-0.0000 (0.0000),-0.0000 (0.0000)


,N,R2
(1),257.0,0.004103
(2),257.0,0.495085
(6),257.0,0.496173


                            OLS Regression Results                            
Dep. Variable:     D_censuspop9010_cp   R-squared:                       0.004
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.209
Date:                Fri, 24 Apr 2026   Prob (F-statistic):              0.282
Time:                        17:53:22   Log-Likelihood:                -60.497
No. Observations:                 257   AIC:                             125.0
Df Residuals:                     255   BIC:                             132.1
Df Model:                           1                                         
Covariance Type:              cluster                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept              0.3723      0

# Question 2 - Analysis

### Regression (1) — Roads Only, No Controls

The coefficient on `all_road_2010_rays` is **+0.0097** (SE = 0.0088, p = 0.272), which is positive but statistically insignificant. The model's R² of 0.004 confirms that radial roads alone <u>explain virtually none of the variation</u> in core-city population growth over 1990–2010. At face value this suggests <u>no relationship</u> between roads and urban growth, but the absence of controls means this estimate is almost certainly confounded — faster-growing cities also tend to build more roads, which would bias the coefficient upward.

---

### Regression (2) — Controls + ΔPref Pop

Adding prefecture-level population growth (`D_censuspop9010_pref`) and baseline city characteristics <u>dramatically changes</u> the picture. The roads coefficient **flips sign** to **−0.0104** (SE = 0.0048), now statistically significant at the 5% level, and R² jumps to 0.495. The sign reversal reflects what the controls are doing: <u>once we hold prefecture-wide growth constant, more radial roads are associated with the core city capturing a *smaller share* of that growth.</u> This is an early signal of the paper's central decentralization finding. Roads appear to draw population toward the periphery rather than concentrating it in the urban core. Among the baseline controls, larger initial core-city area is associated with slower subsequent core growth (consistent with mean reversion), while being a provincial capital is associated with faster growth.

---

### Regression (6) — Controls + ΔPref Pop + Ring Road Indicator

Including the ring-road dummy (`po_s_all_road_2010_ringxda`) <u>changes almost nothing</u>. The radial roads coefficient holds at **−0.0098** (SE = 0.0053) and the ring road indicator is small and insignificant (+0.025, SE = 0.033). <u>R² barely moves from 0.495 to 0.496.</u> In the OLS framework, ring roads <u>do not appear to independently explain core-city population growth beyond what radial roads already capture</u>.

---

### Comparison to Table 3 (Baum-Snow et al., 2017)

Our replicated coefficients match the published Table 3 results closely across all three specifications.

| Specification | Our Estimate | Paper (Table 3) |
|---|---|---|
| (1) `all_road_2010_rays` | +0.0097 (0.0088) | +0.0097 (0.0088) |
| (2) `all_road_2010_rays` | −0.0104 (0.0048) | −0.0114 (0.0050) |
| (6) `all_road_2010_rays` | −0.0098 (0.0053) | −0.0108 (0.0055) |

Column (1) is an exact match. Columns (2) and (6) show only minor differences. My coefficient on radial highways is slightly less relative to the paper's, likely reflecting small differences in sample construction or variable handling. The direction, magnitude, and significance level are consistent in all three cases, and the core finding replicates cleanly: once prefecture-wide population growth and baseline controls are included, more radial highways are associated with **slower core-city population growth**, <u>confirming the paper's decentralization narrative.</u>

---

# Question 3 — Instrumental Variables (2SLS)

In the previous question, we estimated the relationship between road infrastructure and population growth using **ordinary least squares (OLS)**.  
However, OLS estimates may be **biased** if roads were built *because* cities were already growing.  
To address this potential **endogeneity**, we now estimate **instrumental-variables (IV)** regressions using **two-stage least squares (2SLS)**.

---

## 3.1 The IV setup

In a 2SLS model we distinguish among:
- **Dependent variable** – `y`
- **Exogenous regressors** – `x`
- **Endogenous regressors** – `z`
- **Instruments** – `w`

Formally:

$$
y = \alpha + \beta z + x'\gamma + \varepsilon, \qquad
z = \pi_0 + \pi_1 w + x'\rho + u
$$

The key requirement is that instruments `w` are:
1. **Relevant** — correlated with the endogenous variable `z`, and  
2. **Exogenous** — uncorrelated with the structural error \(\varepsilon\).

---

## 3.2 Using `IV2SLS` in Python

We use the `linearmodels` package, which implements 2SLS estimation as:

```python
from linearmodels.iv import IV2SLS
import statsmodels.api as sm

# Example setup
y = df["y"]                       # dependent variable
exog = sm.add_constant(df[["x1", "x2"]])   # exogenous regressors (with constant)
endog = df[["z1"]]                # endogenous variable(s)
instr = df[["w1"]]                # instrument(s)

# Fit the IV model with clustered standard errors
iv_model = IV2SLS(y, exog, endog, instr).fit(
    cov_type="clustered", clusters=df["cluster_id"]
)

# Display both second stage and first stage results
print(iv_model.summary)
print(iv_model.first_stage.summary)
```

---

## 3.3 Tasks

1. **Estimate three IV specifications** that correspond to the OLS regressions from Question 2.  
   - Treat modern road measures as **endogenous** and instrument them using **historical road variables**.  
   - Remember to **cluster standard errors by province**.

2. **Print both outputs** for each model:  
   - The **second-stage regression results**  
     ```python
     print(iv_model.summary)
     ```  
   - The **first-stage regression results**  
     ```python
     print(iv_model.first_stage.summary)
     ```

3. **Assess instrument strength.**  
   - Examine the **first-stage F-statistics** reported at the bottom of each first-stage table.  
   - Are your instruments **strong** (F ≥ 10) or **weak** (F < 10)?  
   - Briefly comment on what that implies for your IV estimates.

4. **Compare the 2SLS and OLS results.**  
   - How do the **coefficients** on road variables differ between OLS and IV?  
   - Does the IV estimate suggest that OLS **overstated** or **understated** the causal effect of roads on urban population growth?  
   - What does this tell you about the **direction of endogeneity bias** in the OLS regressions?


In [26]:
# QUESTION 3 — IV (2SLS) Regressions (Table 4: Columns 1, 2, and 6)

base_controls = [
    "larea_cc90",
    "larea_pf05",
    "province_capitalPlus",
    "lcensuspop1982_pref",
    "frac_highedu1982_pref",
    "share_emp_man1982_pref",
    # "ruralMigp00",
]
def fit_iv_clustered(endog_vars, instr_vars, exog_vars, data, cluster_col, cov_type="clustered"):
    data = data.dropna(subset=[y_var] + endog_vars + instr_vars + exog_vars + [cluster_col])
    y     = data[y_var]
    exog  = sm.add_constant(data[exog_vars]) if exog_vars else sm.add_constant(pd.Series(np.ones(len(data)), index=data.index, name="const"))
    endog = data[endog_vars]
    instr = data[instr_vars]
    fit_kwargs = {"cov_type": cov_type}
    if cov_type == "clustered":
        fit_kwargs["clusters"] = data[cluster_col]
    res = IV2SLS(y, exog, endog, instr).fit(**fit_kwargs)
    return res, data[cluster_col]

# (1) Baseline IV: radial roads only, no controls IV: Δln(CC Pop) ~ all_road_2010_rays  (IV = road_1962_rays)
iv_1, _ = fit_iv_clustered(
    endog_vars=["all_road_2010_rays"],
    instr_vars=["road_1962_rays"],
    exog_vars=[],
    data=df,
    cluster_col=cluster_var,
)

# (2) Add base controls + ΔPref Pop (also endogenous, instrumented by rail_1962_rays)
iv_2, _ = fit_iv_clustered(
    endog_vars=["all_road_2010_rays", "D_censuspop9010_pref"],
    instr_vars=["road_1962_rays", "rail_1962_rays"],
    exog_vars=base_controls,
    data=df,
    cluster_col=cluster_var,
)

# (6) Add ring road indicator (also endogenous, instrumented by po_s_road_1962_ringxda)
iv_6, _ = fit_iv_clustered(
    endog_vars=["all_road_2010_rays", "D_censuspop9010_pref", "po_s_all_road_2010_ringxda"],
    instr_vars=["road_1962_rays", "rail_1962_rays", "po_s_road_1962_ringxda"],
    exog_vars=base_controls,
    data=df,
    cluster_col=cluster_var,
)


In [22]:
# ---- Compact comparison table ----
def compact_table_iv(results_by_col: dict, variables: list) -> pd.DataFrame:
    out = pd.DataFrame(index=variables)
    for col_name, res in results_by_col.items():
        coefs = res.params.reindex(variables)
        ses   = res.std_errors.reindex(variables)
        out[col_name] = [
            f"{c:.4f} ({s:.4f})" if (pd.notna(c) and pd.notna(s)) else ""
            for c, s in zip(coefs, ses)
        ]
    return out

iv_results = {"(1)": iv_1, "(2)": iv_2, "(6)": iv_6}

vars_to_show = [
    "all_road_2010_rays",
    "D_censuspop9010_pref",
    "po_s_all_road_2010_ringxda",
    *base_controls,
    "const",
]

tbl_iv = compact_table_iv(iv_results, vars_to_show)

diag_iv = pd.DataFrame({
    k: {"N": int(v.nobs), "R2": float(v.rsquared)}
    for k, v in iv_results.items()
}).T

# ---- Display Results ----
display(tbl_iv)
display(diag_iv)

# ---- Full output with first-stage diagnostics ----
for col, res in iv_results.items():
    print(f"\n{'='*60}")
    print(f"  IV SECOND STAGE — Specification {col}")
    print(f"{'='*60}")
    print(res.summary)
    print(f"\n--- First Stage: Specification {col} ---")
    print(res.first_stage.summary)

,(1),(2),(6)
all_road_2010_rays,-0.0067 (0.0186),-0.0451 (0.0227),-0.0584 (0.0239)
D_censuspop9010_pref,,0.3854 (2.1205),0.8026 (1.9995)
po_s_all_road_2010_ringxda,,,-0.2499 (0.1433)
larea_cc90,,-0.1248 (0.0464),-0.1653 (0.0661)
larea_pf05,,0.0233 (0.1395),0.0583 (0.1209)
province_capitalPlus,,0.2705 (0.4712),0.2168 (0.4621)
lcensuspop1982_pref,,0.0887 (0.1370),0.1399 (0.1145)
frac_highedu1982_pref,,-0.1935 (1.1326),-0.2594 (1.0320)
share_emp_man1982_pref,,-0.1611 (0.7340),-0.2962 (0.7252)
const,0.4349 (0.0971),-0.1048 (3.6366),-0.8108 (3.2793)


,N,R2
(1),257.0,-0.007678
(2),257.0,0.350581
(6),257.0,0.292818



  IV SECOND STAGE — Specification (1)
                          IV-2SLS Estimation Summary                          
Dep. Variable:     D_censuspop9010_cp   R-squared:                     -0.0077
Estimator:                    IV-2SLS   Adj. R-squared:                -0.0116
No. Observations:                 257   F-statistic:                    0.1311
Date:                Fri, Apr 24 2026   P-value (F-stat)                0.7173
Time:                        17:26:19   Distribution:                  chi2(1)
Cov. Estimator:             clustered                                         
                                                                              
                                 Parameter Estimates                                  
                    Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
--------------------------------------------------------------------------------------
const                  0.4349     0.0971     4.4780     0.0000      

## Answer for 3.3

### 3) Instrument strength
The first-stage partial F-statistics vary considerably across specifications and endogenous variables:
| Spec | Endogenous Variable | Partial F-stat | Strong? |
|---|---|---|---|
| (1) | `all_road_2010_rays` | 37.83 | ✓ Yes |
| (2) | `all_road_2010_rays` | 21.27 | ✓ Yes |
| (2) | `D_censuspop9010_pref` | 1.00 | ✗ No |
| (6) | `all_road_2010_rays` | 21.50 | ✓ Yes |
| (6) | `D_censuspop9010_pref` | 2.85 | ✗ No |
| (6) | `po_s_all_road_2010_ringxda` | 23.59 | ✓ Yes |

The instrument for radial roads (`road_1962_rays`) is strong across all specifications, with F-statistics well above the conventional threshold of 10. 
The ring road instrument (`po_s_road_1962_ringxda`) is similarly strong in spec (6). However, the instrument for `D_censuspop9010_pref`, `rail_1962_rays`, is weak in both specs (2) and (6) (✗ No), with F-statistics near 1. 
This means the IV estimates on `D_censuspop9010_pref` itself are unreliable, and its large, insignificant second-stage coefficients (0.39 and 0.80 with standard errors exceeding 2.0) reflect that weakness. 
Fortunately, the variable of primary interest — `all_road_2010_rays` — is well-identified throughout.

---


### 4) OLS vs. IV — coefficient comparison and bias direction

| Spec | OLS | IV (2SLS) |
|---|---|---|
| (1) `all_road_2010_rays` | +0.0097 (0.0088) | −0.0067 (0.0186) |
| (2) `all_road_2010_rays` | −0.0104 (0.0048) | −0.0451 (0.0227) |
| (6) `all_road_2010_rays` | −0.0098 (0.0053) | −0.0584 (0.0239) |

In all three specifications, the IV coefficient on `all_road_2010_rays` is **more negative** than the corresponding OLS estimate. 
This is consistent with OLS suffering from **upward (positive) endogeneity bias**: <u>cities that were already growing fast were more likely to attract road investment</u>, which pushes the OLS coefficient upward. 
Once we instrument with 1962 planned roads, which Baum et.al show are plausibly unrelated to post-1990 growth shocks, the estimated effect of radial highways on core-city population growth becomes substantially larger in magnitude.

**Interpretation:**  
The IV results strengthen the paper's decentralization finding. 
A one-unit increase in radial road connections is associated with roughly a **4.5–5.8% decrease** in core-city log population growth (specs 2 and 6), compared to only about 1% under OLS. 
<u>OLS was masking the true decentralizing effect of roads</u> because road-building was endogenously concentrated in already-thriving cities. 
After instrumenting, we see that <u>roads genuinely pull population away from urban cores and toward the prefecture periphery</u>. 
This finding that holds up even after controlling for prefecture-wide growth and ring road infrastructure.


# Question 4 — Standard Error Sensitivity Analysis

In applied econometrics, the choice of **standard error calculation** can significantly affect statistical inference. Different assumptions about the error structure lead to different standard error estimates, which in turn affect p-values and conclusions about statistical significance.

In this question, you will re-estimate **specification (6)** from Question 3 (the radial+ring IV regression) using **three different standard error assumptions**:

1. **Classical standard errors** — Assume errors are independent and identically distributed (IID)  
2. **HC1 robust standard errors** — Allow for heteroskedasticity but assume independence  
3. **Clustered standard errors** — Allow for correlation within provinces (as used in main analysis)

Focus your analysis on the coefficient for **`all_road_2010_rays`** only.

---

## 4.1 Tasks

1. **Re-estimate specification (6)** using each of the three standard error types above
2. **Report the three standard errors** for `all_road_2010_rays` and note the significance level under each assumption
3. **Identify which standard error assumption is most conservative** (provides largest standard errors)
4. **Provide conclusions** based on this analysis about the appropriateness of different standard error assumptions

In [33]:
# QUESTION 4 — Standard Error Sensitivity Analysis
# Focus on specification (6) and all_road_2010_rays variable only

# Set up the IV regression (specification 6: radial+ring from Question 3)
controls = [
    "larea_cc90",
    "larea_pf05", 
    "province_capitalPlus",
    "lcensuspop1982_pref",
    "frac_highedu1982_pref",
    "share_emp_man1982_pref",
]
target_var = "all_road_2010_rays"
endog_6 = ["all_road_2010_rays", "D_censuspop9010_pref", "po_s_all_road_2010_ringxda"]
instr_6 = ["road_1962_rays", "rail_1962_rays", "po_s_road_1962_ringxda"]

vars_to_show = [
    # target_var,
    *endog_6,
    # *instr_6,
    *controls,
    "const",
]

# 1) Classical standard errors (IID assumption)
iv_classical, _ = fit_iv_clustered(endog_6, instr_6, controls, df, cluster_var, cov_type="unadjusted")
# 2) HC1 robust standard errors (heteroskedasticity-robust)
iv_robust, _    = fit_iv_clustered(endog_6, instr_6, controls, df, cluster_var, cov_type="robust")

# 3) Clustered standard errors (by province)
iv_clustered, _ = fit_iv_clustered(endog_6, instr_6, controls, df, cluster_var, cov_type="clustered")

iv_results = {"Classical": iv_classical, "Robust": iv_robust, "Clustered": iv_clustered}
tbl_iv = compact_table_iv(iv_results, vars_to_show)
display(tbl_iv)

# Summary
def se_summary(label, res, var):
    coef  = res.params[var]
    se    = res.std_errors[var]
    pval  = res.pvalues[var]
    stars = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.10 else ""
    print(f"{label:<30} coef={coef:.4f}  SE={se:.4f}  p={pval:.4f}  {stars}")

# Extract and print results
print(f"SE Sensitivity — '{target_var}'\n")
se_summary("1) Classical (IID)",      iv_classical, target_var)
se_summary("2) HC1 Robust",           iv_robust,    target_var)
se_summary("3) Clustered (province)", iv_clustered, target_var)

,Classical,Robust,Clustered
all_road_2010_rays,-0.0584 (0.0338),-0.0584 (0.0317),-0.0584 (0.0239)
D_censuspop9010_pref,0.8026 (1.6536),0.8026 (1.7045),0.8026 (1.9995)
po_s_all_road_2010_ringxda,-0.2499 (0.1763),-0.2499 (0.1599),-0.2499 (0.1433)
larea_cc90,-0.1653 (0.0566),-0.1653 (0.0582),-0.1653 (0.0661)
larea_pf05,0.0583 (0.1094),0.0583 (0.1115),0.0583 (0.1209)
province_capitalPlus,0.2168 (0.3686),0.2168 (0.3898),0.2168 (0.4621)
lcensuspop1982_pref,0.1399 (0.0988),0.1399 (0.0977),0.1399 (0.1145)
frac_highedu1982_pref,-0.2594 (0.8537),-0.2594 (0.8127),-0.2594 (1.0320)
share_emp_man1982_pref,-0.2962 (0.5843),-0.2962 (0.5967),-0.2962 (0.7252)
const,-0.8108 (2.7263),-0.8108 (2.8200),-0.8108 (3.2793)


SE Sensitivity — 'all_road_2010_rays'

1) Classical (IID)             coef=-0.0584  SE=0.0338  p=0.0844  *
2) HC1 Robust                  coef=-0.0584  SE=0.0317  p=0.0659  *
3) Clustered (province)        coef=-0.0584  SE=0.0239  p=0.0145  **


## Answer to 4.1 Task 4
* Classical SEs are the smallest when errors are heteroskedastic or correlated, because they assume IID and "undercount" the true variability.
* Robust (HC1) SEs are larger than classical when heteroskedasticity is present because residuals from high-variance observations inflate the uncertainty estimate once relaxing a single variance assumption across all observations.
* Clustered SEs are larger than robust when there is positive within-cluster correlation in the errors (common case in geographic/regional data).


| SE Type | SE | p-value | Significance |
|---|---|---|---|
| 1) Classical (IID) | 0.0338 | 0.084 | * (10%) |
| 2) HC1 Robust | 0.0317 | 0.066 | * (10%) |
| 3) Clustered (province) | 0.0239 | 0.015 | ** (5%) |

The conventional expectation is classical < robust < clustered, reflecting progressively relaxed assumptions about error structure.
For `all_road_2010_rays` the ordering is reversed. 
However, clustered SEs are the smallest and classical the largest. This is not the norm: looking across the full set of coefficients, most variables (e.g. `D_censuspop9010_pref`) follow the expected ordering.
This is somewhat surprising because clustering is usually expected to be the most conservative because it allows for within-group error correlation. 
The anomaly for `all_road_2010_rays` specifically suggests that within-province variation in radial road exposure is actually quite heterogeneous.
That is, cities in the same province don't share a common road shock, so clustering soaks up less unexplained variance than it adds. 
In other words, the within-cluster error correlation for this variable is near zero or negative, which tightens rather than inflates the clustered SEs.

Regardless of this ordering, **clustered standard errors remain the most appropriate choice**. 
Road investment decisions are made at the provincial level in China, meaning unobserved policy and funding shocks are shared within provinces. 
Failing to account for that within-cluster correlation, even when it is small, would misrepresent the true error structure of the data. 
That classical and robust SEs happen to be larger here is a sample-specific outcome, not a reason to prefer them.

Regardless of which assumption is used, `all_road_2010_rays` remains statistically significant across all three: at 10% under classical and HC1, and at 5% under clustering. 
The bottom line: radial roads reduce core-city population growth is robust to the choice of standard error assumption.